# Ders 01 - AI Ajanlarına Giriş

**Yeni Başlayanlar için AI Ajanları** kursunun ilk dersine hoş geldiniz!

**AI ajanı**, büyük bir dil modelini (LLM) akıl yürütme motoru olarak kullanan ve gerçek dünyada *eylemler* yapabilen — API çağırma, veritabanı sorgulama veya kod çalıştırma — bir programdır ve kullanıcı adına bir hedefi gerçekleştirebilir.

Bu defterde ilk ajanınızı oluşturacaksınız: tatil destinasyonları öneren bir **Seyahat Ajanı**. Bu süreçte nasıl yapacağınızı öğreneceksiniz:

1. **Microsoft Agent Framework** kullanarak Azure AI Foundry Agent Servisi'ne bağlanmak.
2. Ajan’a çağırabileceği bir **araç** — basit bir Python fonksiyonu — vermek.
3. Ajanı çalıştırmak ve yanıtını incelemek.
4. Ajanın yanıtını token token akış olarak almak.


## Kurulum

Bu not defterini çalıştırmadan önce, lütfen şunlara sahip olduğunuzdan emin olun:

1. **Yayınlanmış bir sohbet modeli bulunan bir Azure AI Foundry projesi** (örneğin `gpt-4o-mini`).
2. **Azure CLI ile giriş yapılmış olmalı** — terminalinizde `az login` komutunu çalıştırın.
3. **Gerekli ortam değişkenleri ayarlanmış olmalı:**
   - `AZURE_AI_PROJECT_ENDPOINT` — Azure AI Foundry projenizin uç noktası.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — yayınlanan modelinizin adı.

Aşağıdaki hücre, ihtiyaç duyduğunuz Python paketlerini yükler.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## İlk Temsilcinizi Oluşturma

Bir temsilcinin iki şeye ihtiyacı vardır:

- Ona *kim olduğunu* ve *nasıl davranacağını* söyleyen **Talimatlar** (bir sistem istemi).
- Temsilcinin bilgi almak veya işlem yapmak için çağırabileceği `@tool` ile dekore edilmiş Python fonksiyonları olan **Araçlar**.

Aşağıda popüler tatil destinasyonlarının bir listesini döndüren basit bir araç tanımlıyoruz. Temsilci, kullanıcı seyahat önerileri istediğinde bu aracı kullanacaktır.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Akış Yanıtları

Daha etkileşimli bir deneyim için aracının yanıtını **akış** halinde alabilirsiniz. Tam yanıtı beklemek yerine, aracı oluşturdukça metin parçalarını sunar. Bu, çıktıyı gerçek zamanlı göstermek istediğiniz sohbet arayüzlerinde özellikle kullanışlıdır.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Özet

Bu derste şunları öğrendiniz:

- `AzureAIProjectAgentProvider` aracılığıyla Azure AI Foundry Agent Service'e bağlanan bir **sağlayıcı oluşturmayı**.
- Temsilcinin Python işlevlerinizi çağırabilmesi için `@tool` dekoratörü kullanarak bir **araç tanımlamayı**.
- Kullanıcı mesajı ile temsilciyi **çalıştırmayı** ve yanıtını yazdırmayı.
- Gerçek zamanlı çıktı için yanıtları **akış halinde almayı**.

Bir sonraki derste temsilci çerçevelerini daha derinlemesine keşfedecek ve temsilcilere daha güçlü araçlar ile çok adımlı akıl yürütme yeteneklerini nasıl sağlayacağımızı öğreneceğiz.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:  
Bu belge, AI çeviri servisi [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba gösterilse de, otomatik çevirilerin hatalar veya yanlışlıklar içerebileceğini lütfen unutmayınız. Orijinal belge, kendi dilinde yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı nedeniyle ortaya çıkabilecek yanlış anlamalar veya yanlış yorumlamalar konusunda sorumluluk kabul edilmemektedir.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
